# Arioso Demo: Unified AI Music Generation

Arioso provides a single `generate()` interface across 20 music generation platforms.

```
pip install arioso
```

## Platform overview

In [ ]:
import arioso

# See all available platforms
platforms = arioso.list_platforms()
print(f"{len(platforms)} platforms available:\n")

for name in sorted(platforms):
    info = arioso.get_platform_info(name)
    tier = info.get("tier", "?")
    access = info.get("access_type", "?")
    display = info.get("display_name", name)
    print(f"  {name:16s}  {display:40s}  [{access}]")

---

## 1. MusicGen (local, no API key needed)

**Setup:** `pip install arioso[musicgen-transformers]` (or `pip install arioso[musicgen]` for audiocraft)

Runs locally on your machine. Uses HuggingFace transformers or Meta's audiocraft.
No GPU required (but recommended for speed).

In [ ]:
import arioso

song = arioso.generate(
    "upbeat jazz piano with walking bass",
    platform="musicgen",
    duration=8,
    temperature=1.0,
    guidance=3.0,
    model="facebook/musicgen-small",
)

print(f"Status: {song.status}")
print(f"Sample rate: {song.sample_rate} Hz")
print(f"Audio shape: {song.audio_array.shape}")

# To play in notebook:
# from IPython.display import Audio
# Audio(song.audio_array, rate=song.sample_rate)

---

## 2. ElevenLabs Music (REST API)

**Setup:** `pip install arioso[elevenlabs]`

**Auth:** Set `ELEVENLABS_API_KEY` env var. Get a key at https://elevenlabs.io

Best-documented music API. Supports lyrics, song structure, streaming.

In [ ]:
import arioso

song = arioso.generate(
    "cinematic orchestral trailer music, epic and dramatic",
    platform="elevenlabs",
    duration=30,
    instrumental=True,
    output_format="mp3_44100_128",
)

print(f"Status: {song.status}")
print(f"Audio bytes: {len(song.audio_bytes)} bytes")

# Save to file
# with open("epic_trailer.mp3", "wb") as f:
#     f.write(song.audio_bytes)

---

## 3. Suno (via sunoapi.org)

**Setup:** `pip install arioso[sunoapi]`

**Auth:** Set `SUNO_API_KEY` (Bearer token from sunoapi.org) and `SUNO_CALLBACK_URL`.

Best-in-class vocal quality. Async generation with polling.

In [ ]:
import arioso

# Generate (returns pending songs immediately)
songs = arioso.generate_many(
    "summer reggae vibes",
    platform="sunoapi",
    genre="reggae, tropical",
    instrumental=True,
    model="V4_5",
    callback_url="https://your-server.com/webhook",
)

print(songs[0].status)  # 'pending'

# Poll until complete
updated = arioso.check_status(songs[0])
for song in updated:
    if song.status == "complete":
        print(song.audio_url)

# Download audio bytes
song_with_audio = arioso.fetch_audio(updated[0])
# with open("reggae.mp3", "wb") as f:
#     f.write(song_with_audio.audio_bytes)

---

## 4. Stable Audio Open (local, diffusers)

**Setup:** `pip install arioso[stable-audio]`

No API key needed. Runs locally with HuggingFace Diffusers. GPU recommended.
Max 47 seconds. Supports negative prompts and seed control.

In [ ]:
import arioso

song = arioso.generate(
    "ambient electronic soundscape, dreamy pads",
    platform="stable_audio",
    negative_prompt="drums, vocals, distortion",
    duration=10.0,
    num_steps=100,  # 200 is default; lower = faster
    guidance=7.0,
    seed=42,
)

print(f"Status: {song.status}")
print(f"Sample rate: {song.sample_rate} Hz")
print(f"Audio shape: {song.audio_array.shape}")

---

## 5. Google Lyria 2 (Vertex AI)

**Setup:** `pip install arioso[lyria2]`

**Auth:** Set `GOOGLE_CLOUD_PROJECT` and authenticate via `gcloud auth application-default login`,
or set `GOOGLE_CLOUD_API_KEY`. Optionally set `GOOGLE_CLOUD_LOCATION` (default: us-central1).

Fixed 30-second outputs at 48 kHz stereo. Supports negative prompts and batch generation.

In [ ]:
import arioso

song = arioso.generate(
    "lo-fi hip hop beats, chill study music",
    platform="lyria2",
    negative_prompt="heavy metal, distortion",
    seed=12345,
)

print(f"Status: {song.status}")
print(f"Duration: {song.audio.duration_seconds}s")
print(f"Audio: {len(song.audio_bytes)} bytes (WAV at {song.sample_rate} Hz)")

---

## 6. Google Lyria RealTime (streaming, most parameter-rich)

**Setup:** `pip install arioso[lyria-rt]`

**Auth:** Set `GOOGLE_API_KEY`. Get one at https://aistudio.google.com/apikey

The only platform with true real-time streaming + direct BPM, key, density, and brightness controls.
Outputs continuous 48 kHz stereo PCM. Closest to gesture-controlled music.

In [ ]:
import arioso

song = arioso.generate(
    "funky disco groove with brass section",
    platform="lyria_rt",
    duration=10.0,       # collect 10 seconds of streaming audio
    bpm=120,             # beats per minute (60-200)
    key="C_MAJOR",       # musical scale
    energy=0.7,          # density (0-1)
    brightness=0.6,      # spectral brightness (0-1)
    guidance=4.0,        # guidance strength (0-6)
    temperature=1.1,     # sampling temperature (0-3)
    top_k=40,            # top-k sampling (1-1000)
    seed=42,
)

print(f"Status: {song.status}")
print(f"PCM audio: {len(song.audio_bytes)} bytes at {song.sample_rate} Hz")

# Convert PCM to playable audio:
# import numpy as np
# pcm = np.frombuffer(song.audio_bytes, dtype=np.int16)
# from IPython.display import Audio
# Audio(pcm.astype(np.float32) / 32768, rate=48000)

---

## 7. Mubert (REST API)

**Setup:** `pip install arioso[mubert]`

**Auth:** Set `MUBERT_PAT` (Personal Access Token). Get one at https://mubert.com (API plan: $49/mo).

Up to 25 minutes of music. Supports energy/intensity levels.

In [ ]:
import arioso

song = arioso.generate(
    "relaxing background music for a cafe",
    platform="mubert",
    duration=60,          # 15-1500 seconds
    output_format="mp3",
    bitrate=320,
    energy=0.3,           # low energy (coerced to "low")
)

print(f"Status: {song.status}")
print(f"Audio URL: {song.audio_url}")

---

## 8. Beatoven.ai (REST API)

**Setup:** `pip install arioso[beatoven]`

**Auth:** Set `BEATOVEN_API_KEY`. Get 50 free credits at https://beatoven.ai/api

Supports looping, creativity control, and refinement steps.

In [ ]:
import arioso

song = arioso.generate(
    "uplifting corporate background music",
    platform="beatoven",
    duration=30,          # 5-150 seconds
    guidance=16,          # creativity (native: creativity)
    num_steps=100,        # refinement steps
    looping=True,         # generate loopable audio
    seed=42,
)

print(f"Status: {song.status}")
print(f"Audio URL: {song.audio_url}")

---

## 9. Loudly (REST API)

**Setup:** `pip install arioso[loudly]`

**Auth:** Set `LOUDLY_API_KEY`. Free tier available at https://loudly.com/developers

Most structured REST API: genre (50+), BPM, key, energy, instruments (up to 7), song structure.

In [ ]:
import arioso

song = arioso.generate(
    "energetic workout music",
    platform="loudly",
    duration=60,
    genre="EDM",
    bpm=140,
    key="A minor",
    energy=0.9,
    instruments=["synth", "drums", "bass"],
)

print(f"Status: {song.status}")
print(f"Audio URL: {song.audio_url}")

---

## 10. Jen (REST API)

**Setup:** `pip install arioso[jen]`

**Auth:** Set `JEN_API_KEY`. 300 free credits at https://api.jenmusic.ai/docs/getting-started

48 kHz stereo. Features STRUCTUR3 for song structure and R3FILL for inpainting.

In [ ]:
import arioso

song = arioso.generate(
    "acoustic folk guitar with gentle fingerpicking",
    platform="jen",
    duration=30,
    output_format="wav",
)

print(f"Status: {song.status}")
print(f"Audio URL: {song.audio_url}")

---

## 11. YuE (lyrics-to-full-song, fal.ai or local)

**Setup:** `pip install arioso[yue]`

**Auth (fal.ai):** Set `FAL_KEY`. If not set, falls back to local CLI (requires YuE repo + GPU).

Unique: generates **separate vocal and instrumental tracks** natively.
Supports multilingual lyrics with structural tags (`[verse]`, `[chorus]`, `[outro]`).

In [ ]:
import arioso

song = arioso.generate(
    "indie rock, male vocals, upbeat",  # genre description
    platform="yue",
    lyrics="""[verse]
Walking down the road at dawn
Coffee in my hand, the world moves on
[chorus]
This is the day, the day we start again
Nothing can stop us now, my friend
[outro]
Start again...
""",
    duration=60,              # ~2 segments of 30s each
    max_tokens=3000,
    repetition_penalty=1.1,
)

print(f"Status: {song.status}")
print(f"Audio URL: {song.audio_url}")

---

## 12. Harmonai / Dance Diffusion (local, unconditional)

**Setup:** `pip install arioso[harmonai]`

No API key needed. **Unconditional** generation (no text prompts). Experimental quality.
Good for generating short audio textures and samples.

In [ ]:
import arioso

# Note: prompt is ignored (unconditional model)
song = arioso.generate(
    "",  # no prompt needed
    platform="harmonai",
    num_steps=100,
    seed=42,
)

print(f"Status: {song.status}")
print(f"Sample rate: {song.sample_rate} Hz")
print(f"Audio shape: {song.audio_array.shape}")

---

## 13. Riffusion (local, spectrogram-based)

**Setup:** `pip install arioso[riffusion]`

No API key needed. Generates music via spectrogram image synthesis.
Uses either the `riffusion` library (if installed) or `diffusers` fallback with Griffin-Lim reconstruction.

In [ ]:
import arioso

song = arioso.generate(
    "reggae guitar riff with tropical vibes",
    platform="riffusion",
    negative_prompt="noise, distortion",
    guidance=7.0,
    num_steps=50,
    seed=42,
)

print(f"Status: {song.status}")
print(f"Sample rate: {song.sample_rate} Hz")
print(f"Backend: {song.metadata.get('backend', '?')}")

---

## 14. Udio (unofficial wrapper)

**Setup:** `pip install arioso[udio]`

**Auth:** Set `UDIO_AUTH_COOKIE` (browser cookie from udio.com login).

Note: Uses the unofficial `udio-wrapper` package. TOS gray area.

In [ ]:
import arioso

song = arioso.generate(
    "smooth R&B with soulful vocals",
    platform="udio",
    lyrics="Baby, we got time tonight / Under the city lights so bright",
    seed=42,
)

print(f"Status: {song.status}")
print(f"Audio URL: {song.audio_url}")

---

## Platforms without public APIs

The following 6 platforms are registered in arioso for configuration/introspection purposes,
but **do not have programmatic APIs**. Calling `generate()` on them raises `NotImplementedError`
with guidance on how to use each platform's native interface.

| Platform | Access | Notes |
|----------|--------|-------|
| **AIVA** | Web app | 250+ style presets, orchestral focus. aiva.ai |
| **ACE Studio** | DAW plugin | 140+ AI voice models for vocal synthesis. acestudio.ai |
| **Boomy** | Web app (enterprise API) | Auto Vocal, distribution to streaming. boomy.com |
| **Soundraw** | Web app (enterprise B2B API) | Customizable loops. soundraw.io |
| **CassetteAI** | Web app | Simple prompt-and-duration. cassetteai.com |
| **Musicfy** | Web app | Voice cloning, pitch shifting. musicfy.lol |

In [ ]:
# You can still inspect their configurations:
info = arioso.get_platform_info("aiva")
print(f"AIVA: {info['display_name']}")
print(f"  Website: {info['website']}")
print(f"  Supported affordances: {info['supported_affordances']}")
print(f"  Notes: {info['notes']}")

# Calling generate() raises NotImplementedError with guidance:
try:
    arioso.generate("orchestral", platform="aiva")
except NotImplementedError as e:
    print(f"\n  Error: {e}")

---

## Quick setup reference

### Environment variables

```bash
# REST API platforms
export ELEVENLABS_API_KEY="..."       # elevenlabs.io
export SUNO_API_KEY="..."             # sunoapi.org
export SUNO_CALLBACK_URL="..."        # your webhook URL
export GOOGLE_API_KEY="..."           # aistudio.google.com/apikey (Lyria RT)
export GOOGLE_CLOUD_PROJECT="..."     # Google Cloud project (Lyria 2)
export MUBERT_PAT="..."               # mubert.com API plan
export BEATOVEN_API_KEY="..."         # beatoven.ai/api
export LOUDLY_API_KEY="..."           # loudly.com/developers
export JEN_API_KEY="..."              # api.jenmusic.ai

# Python-lib platforms with cloud backends
export FAL_KEY="..."                  # fal.ai (for YuE)
export UDIO_AUTH_COOKIE="..."         # browser cookie (unofficial)
```

### Install extras

```bash
# Individual platforms
pip install arioso[musicgen-transformers]  # MusicGen via transformers
pip install arioso[stable-audio]           # Stable Audio Open
pip install arioso[lyria-rt]               # Google Lyria RealTime
pip install arioso[yue]                    # YuE via fal.ai

# All REST API platforms
pip install arioso[all-rest]

# Everything
pip install arioso[all]
```

### Platform tiers (by API maturity)

| Tier | Platforms | Notes |
|------|-----------|-------|
| **Tier 1** (production) | ElevenLabs, Lyria 2, MusicGen, Beatoven | Best docs, official SDKs |
| **Tier 2** (usable) | Suno*, Stable Audio, YuE, Mubert, Loudly | *Unofficial wrapper for Suno |
| **Tier 3** (limited) | Udio*, Riffusion, Jen, Harmonai | Experimental or sparse docs |
| **Tier 4** (no API) | AIVA, ACE Studio, Boomy, Soundraw, CassetteAI, Musicfy | Web/desktop only |